# Konwertuje ckpt na nnue a potem ewaluuje

In [ ]:
import torch
import os

# Define paths
CHECKPOINT_PATH = "workdir/version_0/checkpoints/last.ckpt"
EXPORT_NAME = "eval_net.nnue"

# We use the provided serialization script from the repo
!python3 serialize.py {CHECKPOINT_PATH} {EXPORT_NAME}

if os.path.exists(EXPORT_NAME):
    print(f"Successfully converted {CHECKPOINT_PATH} to {EXPORT_NAME}")

In [ ]:
# Paths to your engine binaries
ENGINE_BIN = "./stockfish" # Path to a Stockfish binary that supports NNUE
OPENING_BOOK = "probi_2.0.epd" # You need an opening book for meaningful Elo tests

# Create a small shell script to wrap Stockfish with your custom net
with open("test_engine.sh", "w") as f:
    f.write(f"#!/bin/bash\n")
    f.write(f"{ENGINE_BIN} eval_net.nnue") # Some versions require 'setoption name EvalFile'

!chmod +x test_engine.sh

In [ ]:
# Parameters for the match
GAMES = 100
TC = "10+0.1" # Time control: 10 seconds + 0.1 increment
THREADS = 1

!./cutechess-cli \
  -engine name=NewNet cmd=./stockfish option.EvalFile={EXPORT_NAME} \
  -engine name=Baseline cmd=./stockfish \
  -each proto=uci tc={TC} \
  -games {GAMES} \
  -repeat \
  -openings file={OPENING_BOOK} format=epd order=random \
  -concurrency {THREADS} \
  -pgnout results.pgn

In [ ]:
with open("results.pgn", "r") as f:
    lines = f.readlines()
    # Looking for the cutechess summary line usually at the end
    for line in lines:
        if "Elo difference" in line:
            print(line.strip())